# КОНТУР · обучение детектора RF-DETR в Google Colab (TPU v5e-1)

1. **Один раз** положите `artifacts/kontur-dataset.zip` (≈97 МБ) на Google Диск в папку `kontur/`:
   получится `MyDrive/kontur/kontur-dataset.zip`. Дальше Colab берёт файл с Диска за секунды, а
   загрузка через браузер (`files.upload()`) занимала минуты и начиналась заново при каждом запуске.
2. **Среда выполнения → Сменить среду выполнения → TPU** (в Colab это v5e-1, 16 ГБ HBM).
3. **Среда выполнения → Выполнить всё.** Ноутбук сам определит, TPU под ним или GPU, и подберёт
   настройки обучения под то, что нашёл.
4. Если шаг 2 заменил `torch` (он скажет об этом), ядро перезапустится само — тогда снова
   **Выполнить всё**. Второй раз установка уже ничего не меняет и всё идёт до конца.
5. В конце `kontur-model.zip` ложится на Диск (`MyDrive/kontur/`) и скачивается в браузер.
   Распакуйте его в папку проекта `.models/rfdetr/kontur/` и запустите сервер с `KONTUR_DETECTOR=rfdetr`.

Выборка: открытый набор коррозии Roboflow 100 `corrosion-bi3q3` (CC BY 4.0, снимки с дрона: мосты и
трубы) и снимки учебного стенда КОНТУР с 3D-моделью насоса (норма, течь уплотнения, нет кожуха
муфты). Почти-копии кадров train убраны из test заранее.

## Почему TPU требует других настроек

RF-DETR 1.12.1 обучается на TPU через PyTorch/XLA (экстра `rfdetr[train,xla]`: `torch_xla 2.9`,
только Linux). XLA компилирует граф под конкретные формы тензоров, поэтому всё, что меняет форму от
батча к батчу, у него превращается в пересборку графа:

- **`pad_targets_to`** — число рамок в кадре задаёт форму функции потерь. Без выравнивания каждый
  новый набор счётчиков рамок компилируется заново (у Roboflow на Cloud TPU v6e-1 — 78 компиляций и
  1453 с против 9 и 159 с с выравниванием, issue #1433). Ниже значение считается по самой выборке.
- **`multi_scale=False`** — многомасштабное обучение даёт свой граф на каждый масштаб.
- **`amp_dtype='bf16'`** — на TPU это XLA `bf16-true`, родная точность матричных блоков.
- **`batch_size`** числом, не `'auto'`: автоподбор замеряет память CUDA и на TPU не работает.

Прямого замера «TPU v5e-1 против T4» у нас нет — посмотрите на время первой и второй эпохи в полосе
прогресса: первая включает компиляцию графа, установившуюся скорость показывает вторая.

## Если что-то упало

- `RuntimeError: Only a single TORCH_LIBRARY can be used to register the namespace triton` —
  `torch` подменили под работающим ядром, и его `__init__` выполнился дважды. Лечится только
  перезапуском: **Среда выполнения → Перезапустить сеанс**, потом **Выполнить всё**. Шаг 2 теперь
  делает это сам, но если попали в такое состояние руками — просто перезапустите сеанс.
- `ImportError: ... undefined symbol` при импорте `torch_xla` — минорные версии `torch` и
  `torch_xla` разошлись (расширение `_XLAC` собрано под одну). Шаг 3 ловит это до импорта и
  печатает обе версии.


In [ ]:
# 1. Что за среда выполнения досталась. torch здесь намеренно не импортируется: установка
# ниже может заменить сам torch, а уже загруженный модуль подменой не обновится.
import os
import pathlib
import shutil
import subprocess
import sys

TPU = (any(name.startswith('TPU_') for name in os.environ)
       or pathlib.Path('/dev/accel0').exists() or pathlib.Path('/dev/vfio/0').exists())
GPU = not TPU and shutil.which('nvidia-smi') is not None

print('Python', sys.version.split()[0], '| ядер CPU', os.cpu_count())
if TPU:
    print('TPU:', os.environ.get('TPU_ACCELERATOR_TYPE', 'есть'))
elif GPU:
    print(subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout.strip())
else:
    print('Ускорителя нет — обучение на CPU займёт часы.',
          'Среда выполнения → Сменить среду выполнения → TPU.')


In [ ]:
# 2. Та же версия RF-DETR, что на сервере КОНТУР: чекпоинт должен открываться там без сюрпризов.
# Главное — не заменить torch в уже работающем сеансе: переписанный под ядром torch выполняет
# свой __init__ второй раз, и следующий импорт падает на «Only a single TORCH_LIBRARY can be used
# to register the namespace triton». Поэтому смотрим, что уже стоит, ставим минимум, а если pip
# всё-таки тронул torch — перезапускаем ядро, и после перезапуска ставить будет уже нечего.
import importlib
from importlib.metadata import PackageNotFoundError, version


def installed(package):
    importlib.invalidate_caches()
    try:
        return version(package)
    except PackageNotFoundError:
        return None


BEFORE = {name: installed(name) for name in ('torch', 'torch_xla', 'rfdetr')}
# В среде Colab с TPU torch и torch_xla уже парные (расширение _XLAC собрано под одну минорную
# версию torch) — тогда ставим rfdetr, не трогая их. Экстра xla нужна там, где torch_xla нет
# совсем или он от другой версии torch: она притягивает torch_xla 2.9 и torch 2.9.* вместе.
paired = bool(BEFORE['torch'] and BEFORE['torch_xla']
              and BEFORE['torch'].split('.')[:2] == BEFORE['torch_xla'].split('.')[:2])
EXTRA = 'train,xla' if TPU and not paired else 'train'
print('было:', BEFORE, '→ ставим rfdetr[%s]==1.12.1' % EXTRA)

!pip install -q "rfdetr[{EXTRA}]==1.12.1"

AFTER = {name: installed(name) for name in BEFORE}
print('стало:', AFTER)
if (AFTER['torch'], AFTER['torch_xla']) != (BEFORE['torch'], BEFORE['torch_xla']):
    print('\ntorch заменён в работающем сеансе — перезапускаю ядро.')
    print('Дальше снова Среда выполнения → Выполнить всё: установка не повторится,')
    print('а torch и torch_xla загрузятся уже в чистом процессе.')
    try:
        get_ipython().kernel.do_shutdown(True)
    except Exception:
        os.kill(os.getpid(), 9)


In [ ]:
# 3. Ускоритель виден обучению? Пару torch / torch_xla сверяем по метаданным до импорта: при
# расхождении минорных версий импорт torch_xla падает с нечитаемой ошибкой про undefined symbol.
from importlib.metadata import version

if TPU:
    try:
        torch_minor = version('torch').split('+')[0].split('.')[:2]
        xla_minor = version('torch_xla').split('+')[0].split('.')[:2]
        assert torch_minor == xla_minor, f"torch {version('torch')} и torch_xla {version('torch_xla')} несовместимы"
        os.environ.setdefault('PJRT_DEVICE', 'TPU')
        import torch_xla.core.xla_model as xm
        print('torch', version('torch'), '| torch_xla', version('torch_xla'), '| устройство', xm.xla_device())
    except Exception as error:
        # libtpu (его ставит экстра torch_xla[tpu]) в среде Colab с TPU обычно уже есть.
        print('torch_xla не поднялся:', type(error).__name__, error)
        print('Выполните в новой ячейке:  !pip install -q "torch_xla[tpu]==2.9.*"')
        print('и затем Среда выполнения → Перезапустить сеанс.')
else:
    import torch
    print('torch', torch.__version__, '| CUDA', torch.cuda.is_available(),
          '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'без GPU')


In [ ]:
# 4. Выборка с Google Диска. Файл копируется на локальный диск Colab и распаковывается уже
# оттуда: чтение 1440 мелких файлов напрямую с Диска (FUSE) тормозит каждую эпоху.
import glob
import json
import zipfile

DRIVE = '/content/drive/MyDrive'
DRIVE_DIR = f'{DRIVE}/kontur'            # куда вы положили архив и куда вернётся модель
ZIP = '/content/kontur-dataset.zip'

if not os.path.exists(ZIP):
    from google.colab import drive
    drive.mount('/content/drive')
    found = [path for path in (f'{DRIVE_DIR}/kontur-dataset.zip', f'{DRIVE}/kontur-dataset.zip')
             if os.path.exists(path)] or glob.glob(f'{DRIVE}/*/kontur-dataset.zip')
    if found:
        print('беру с Диска:', found[0])
        shutil.copyfile(found[0], ZIP)
    else:
        # Запасной путь: загрузка через браузер, как было раньше.
        print(f'На Диске нет kontur-dataset.zip — положите его в {DRIVE_DIR}/ или загрузите сейчас.')
        from google.colab import files
        files.upload()

zipfile.ZipFile(ZIP).extractall('/content')
DATASET = '/content/dataset'

PHOTOS, MAX_BOXES = {}, {}
for split in ('train', 'valid', 'test'):
    annotations = json.load(open(f'{DATASET}/{split}/_annotations.coco.json'))
    boxes = {image['id']: 0 for image in annotations['images']}
    for item in annotations['annotations']:
        boxes[item['image_id']] += 1
    PHOTOS[split], MAX_BOXES[split] = len(annotations['images']), max(boxes.values(), default=0)
    print(split, PHOTOS[split], 'снимков,', len(annotations['annotations']),
          'рамок, больше всего в одном кадре —', MAX_BOXES[split])

# Выравниваем цели train до самого «населённого» кадра: больше — лишняя память, меньше — лишние
# рамки отбрасываются с предупреждением.
PAD_TARGETS = max(4, MAX_BOXES['train'])
print('pad_targets_to =', PAD_TARGETS)


In [ ]:
# 5. Обучение. Small — баланс скорости и качества (на сервере ≈58 мс на кадр).
from rfdetr import RFDETRMedium, RFDETRNano, RFDETRSmall

SIZE, EPOCHS, BATCH = 'small', 30, 16
OUTPUT = '/content/kontur'

if TPU:
    # Неполный последний батч — ещё одна форма тензора и ещё одна компиляция графа, поэтому для
    # валидации берём самый крупный делитель её размера (195 кадров → 15; train: 1440 = 16 × 90).
    EVAL_BATCH = next((size for size in range(BATCH, 7, -1) if PHOTOS['valid'] % size == 0), BATCH)
    TUNED = dict(accelerator='tpu', devices=1, batch_size=BATCH, grad_accum_steps=1,
                 eval_batch_size=EVAL_BATCH, multi_scale=False, pad_targets_to=PAD_TARGETS,
                 amp_dtype='bf16', num_workers=min(8, os.cpu_count() or 2))
else:
    # На CUDA автоподбор сам берёт максимальный батч, какой влезает в память, и добивает
    # накоплением градиента до 16 кадров на шаг оптимизатора.
    TUNED = dict(batch_size='auto', num_workers=min(4, os.cpu_count() or 2))
print(TUNED)

model = {'nano': RFDETRNano, 'small': RFDETRSmall, 'medium': RFDETRMedium}[SIZE]()
model.train(dataset_dir=DATASET, epochs=EPOCHS, lr=1e-4, output_dir=OUTPUT,
            progress_bar='tqdm', early_stopping=True, early_stopping_patience=8, **TUNED)


In [ ]:
# 6. Проверка на отложенной части (test), так же, как сервер видит кадр с телефона:
# длинная сторона 640 px, JPEG 70 %, тот же RFDETR.from_checkpoint и predict.
# Считаем на CPU: кадры тут разного размера, а XLA компилировал бы граф под каждый размер заново.
import datetime
import io

from PIL import Image
from rfdetr import RFDETR

CHECKPOINT = f'{OUTPUT}/checkpoint_best_total.pth'
detector = RFDETR.from_checkpoint(CHECKPOINT, device='cpu')
names = detector.class_names
test = json.load(open(f'{DATASET}/test/_annotations.coco.json'))
category = {item['id']: item['name'] for item in test['categories']}
truth = {image['id']: [] for image in test['images']}
for item in test['annotations']:
    x, y, w, h = item['bbox']
    truth[item['image_id']].append((category[item['category_id']], (x, y, x + w, y + h)))

records = []
for image in test['images']:
    photo = Image.open(f"{DATASET}/test/{image['file_name']}").convert('RGB')
    scale = min(1.0, 640 / max(photo.size))
    frame = photo.resize((round(photo.width * scale), round(photo.height * scale)))
    buffer = io.BytesIO()
    frame.save(buffer, 'JPEG', quality=70)
    frame = Image.open(io.BytesIO(buffer.getvalue())).convert('RGB')
    found = detector.predict(frame, threshold=0.05)
    predicted = [(names[int(c)], tuple(v / scale for v in box), float(s))
                 for box, s, c in zip(found.xyxy.tolist(), found.confidence.tolist(), found.class_id.tolist())
                 if 0 <= int(c) < len(names)]
    source = image['file_name'].split('__')[0]
    records.append({'file': image['file_name'], 'source': source, 'truth': truth[image['id']], 'pred': predicted})
print(len(records), 'снимков проверено')


In [ ]:
# 7. Метрики по классам: AP50 (площадь под кривой точность–полнота при IoU ≥ 0.5),
# точность и полнота при выбранном пороге. Порог — наибольшая полнота при точности не ниже 50 %:
# заказчику пропуск дефекта хуже ложной тревоги.
CLASSES = ['frame_corrosion', 'seal_leak_excessive', 'guard_missing']
MIN_PRECISION = 0.5

def iou(a, b):
    x1, y1, x2, y2 = max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union > 0 else 0.0

def match(kind, threshold=0.0):
    """True and false positives in score order, and how many boxes there were to find."""
    hits, total = [], 0
    for record in records:
        boxes = [box for name, box in record['truth'] if name == kind]
        total += len(boxes)
        used = [False] * len(boxes)
        for name, box, score in sorted(record['pred'], key=lambda item: -item[2]):
            if name != kind or score < threshold:
                continue
            best, index = 0.5, -1
            for j, other in enumerate(boxes):
                overlap = iou(box, other)
                if not used[j] and overlap >= best:
                    best, index = overlap, j
            if index >= 0:
                used[index] = True
            hits.append((score, index >= 0))
    hits.sort(key=lambda item: -item[0])
    return hits, total

def ap50(kind):
    hits, total = match(kind)
    if not total:
        return None
    tp = fp = 0
    points = []
    for _, good in hits:
        tp += good
        fp += not good
        points.append((tp / total, tp / (tp + fp)))
    ap, previous = 0.0, 0.0
    for i, (recall, _) in enumerate(points):
        precision = max(p for _, p in points[i:])
        ap += (recall - previous) * precision
        previous = recall
    return ap

def at(threshold, kinds=CLASSES):
    tp = fp = total = 0
    for kind in kinds:
        hits, count = match(kind, threshold)
        tp += sum(good for _, good in hits)
        fp += sum(not good for _, good in hits)
        total += count
    return (tp / total if total else None), (tp / (tp + fp) if tp + fp else None)

candidates = [round(0.05 * step, 2) for step in range(1, 19)]
best = None
for threshold in sorted(candidates, reverse=True):
    recall, precision = at(threshold)
    if precision is not None and precision >= MIN_PRECISION and (best is None or recall > best[1]):
        best = (threshold, recall)
THRESHOLD = best[0] if best else 0.3

def percent(value):
    return '—' if value is None else f'{value * 100:.0f}%'


lines = [f'# RF-DETR {SIZE}, test: {len(records)} снимков', '',
         f'Порог {THRESHOLD} (наибольшая полнота при точности не ниже {MIN_PRECISION:.0%})', '',
         '| Класс | AP50 | Полнота | Точность | Рамок в разметке |', '|---|---|---|---|---|']
for kind in CLASSES:
    recall, precision = at(THRESHOLD, [kind])
    lines.append(f'| {kind} | {percent(ap50(kind))} | {percent(recall)} | {percent(precision)} | {match(kind)[1]} |')
normal = [r for r in records if r['source'] == 'stand' and r['file'].split('__')[1].startswith('normal')]
noisy = sum(any(score >= THRESHOLD for _, _, score in r['pred']) for r in normal)
lines += ['', f'Исправный стенд: рамки хотя бы на одном кадре — {noisy} из {len(normal)} снимков',
          '', 'frame_corrosion — открытый набор Roboflow 100 (снимки с дрона), стенд — рендеры учебного стенда.',
          'Проверка говорит о качестве на этих данных, а не на насосах заказчика.']
report = '\n'.join(lines)
print(report)

In [ ]:
# 8. Упаковка: модель, выбранный порог и отчёт. Архив ложится на Диск — он останется там и
# после того, как среда выполнения отключится. Распакуйте в .models/rfdetr/kontur/ проекта.
os.makedirs('/content/export', exist_ok=True)
shutil.copy(CHECKPOINT, '/content/export/checkpoint_best_total.pth')
json.dump({'threshold': THRESHOLD, 'min_precision': MIN_PRECISION, 'photos': len(records),
           'dataset': 'kontur-dataset.zip', 'evaluated_at': datetime.datetime.now().isoformat(timespec='seconds')},
          open('/content/export/threshold.json', 'w'), ensure_ascii=False, indent=2)
open('/content/export/eval-test.md', 'w').write(report + '\n')
json.dump({'size': SIZE, 'epochs': EPOCHS, 'rfdetr': '1.12.1', 'trained_on': 'Google Colab',
           'accelerator': 'TPU' if TPU else ('GPU' if GPU else 'CPU'), 'tuned': {k: str(v) for k, v in TUNED.items()}},
          open('/content/export/TRAINING.json', 'w'), ensure_ascii=False, indent=2)
shutil.make_archive('/content/kontur-model', 'zip', '/content/export')

if os.path.isdir(DRIVE):
    os.makedirs(DRIVE_DIR, exist_ok=True)
    shutil.copyfile('/content/kontur-model.zip', f'{DRIVE_DIR}/kontur-model.zip')
    print('на Диске:', f'{DRIVE_DIR}/kontur-model.zip')

from google.colab import files
files.download('/content/kontur-model.zip')
